### Import libraries

In [1]:
from pprint import pprint

import grain.python as grain

import jax
import jax.numpy as jnp

import flax.nnx as nnx
import optax

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
# from tokenizers.pre_tokenizers import Whitespace

/home/arjunprasaath/miniconda3/envs/llm_jax/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Architecture

#### Things to build:

- [ x ] Load the dataset
- [ x ] Preprocess the dataset
- [ x ] Dataloader for training (https://google-grain.readthedocs.io/en/latest/tutorials/data_sources/huggingface_dataset_tutorial.html)
- [ x ] Building a Tokenizer (https://huggingface.co/docs/tokenizers/en/quicktour?code=python)
- [ x ] Build a Token embedding and Positional embedding
- [ ] Build Transformer block
    - [ ] MHA block
    - [ ] Feed-Forward block
- [ ] Ouput block
- [ ] Training script
- [ ] Testing script




#### Load the dataset

In [2]:
ds = load_dataset("karpathy/tinystories-gpt4-clean", split="train")

In [3]:
test = ds.select(range(0, 100_000))

In [4]:
test['text'][0]

'Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.\nJack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.\nOnce the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!\nThe more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that.'

In [6]:
# DONT RUN THIS (runs over the 500_000 samples to check the max number of tokens in those samples in a story)
import numpy as np

# Sample 500,000 random stories
sample = ds.shuffle(seed=42).select(range(500_000))
encodings = tokenizer.encode_batch(sample["text"])
lengths = [len(e.ids) for e in encodings]

print(f"Max in sample:    {np.max(lengths)}")
print(f"95th percentile:  {np.percentile(lengths, 95)}")
print(f"99th percentile:  {np.percentile(lengths, 99)}")
print(f"99.9th percentile:{np.percentile(lengths, 99.9)}")


Max in sample:    706
95th percentile:  230.0
99th percentile:  354.0
99.9th percentile:519.0


#### Building the tokenizer

In [54]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(special_tokens=["[PAD]", "[UNK]", "<|startoftext|>", "<|endoftext|>"], vocab_size=10000)


In [55]:
def batch_iterator(batch_size=1):
    tok_dataset = test.select_columns("text")
    for batch in tok_dataset.iter(batch_size):
        yield batch["text"]

In [56]:
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer, length=len(test))

In [57]:
tokenizer.save("data/tokenizer.json")

#### Preprocess the dataset

In [25]:
# adding <|startoftext|> and <|endoftext|> tokens to the stories.
def add_spl_tokens(example):
    example['text'] = "<|startoftext|> " + example['text'] + " <|endoftext|>"
    return example

In [32]:
updated_ds = test.map(add_spl_tokens)

In [31]:
print(updated_ds["text"][0])

<|startoftext|> Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.
Once the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!
The more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that. <|endoftext|>


#### Dataloader

In [5]:
maxlen = 1024

tokenizer = Tokenizer.from_file("data/tokenizer.json")
pad_id = tokenizer.token_to_id("[PAD]")
tokenizer.enable_padding(length=maxlen, pad_id = pad_id, pad_token="[PAD]")
tokenizer.enable_truncation(max_length=maxlen)

In [6]:
# adding <|startoftext|> and <|endoftext|> tokens to the stories.
def add_spl_tokens(example):
    example['text'] = "<|startoftext|> " + example['text'] + " <|endoftext|>"
    return example

def encoding_text(example):
    return tokenizer.encode(example['text']).ids

In [7]:
# Have a look at: https://google-grain.readthedocs.io/en/latest/tutorials/data_sources/huggingface_dataset_tutorial.html
dataset = (
    grain.MapDataset.source(ds)
    .shuffle(seed=40)
    .map(add_spl_tokens)
    .map(encoding_text)
    .map(jnp.asarray)
    .batch(batch_size=3)
)

In [8]:
for i in dataset:
    pprint(i)
    break

array([[   2,    5,  326, ...,    0,    0,    0],
       [   2,    5, 4636, ...,    0,    0,    0],
       [   2,    5, 2272, ...,    0,    0,    0]],
      shape=(3, 1024), dtype=int32)


In [9]:
# Have a look at : https://google-grain.readthedocs.io/en/latest/tutorials/data_loader_tutorial.html#transformations
class ProcessStory(grain.MapTransform):
    def map(self, element):
        text = "<|startoftext|> " + element['text'] + " <|endoftext|>"
        tokenized_text = tokenizer.encode(element['text']).ids
        return jnp.asarray(tokenized_text)

sampler = grain.IndexSampler(
    num_records = len(ds),
    num_epochs = 1,
    shuffle = True,
    seed = 40,
    shard_options = grain.NoSharding()
)

In [10]:
dataloader = grain.DataLoader(
    data_source = ds,
    sampler = sampler,
    operations = [ProcessStory(),
     grain.Batch(batch_size=32, drop_remainder=True)],
     worker_count = 0
)

In [11]:
for batch in dataloader:
    pprint(batch)
    break

array([[ 326, 4506, 1162, ...,    0,    0,    0],
       [4636, 9174, 6919, ...,    0,    0,    0],
       [2272,  442, 1121, ...,    0,    0,    0],
       ...,
       [1293, 1655, 7117, ...,    0,    0,    0],
       [ 419, 4603, 5381, ...,    0,    0,    0],
       [2714, 1248,  111, ...,    0,    0,    0]],
      shape=(32, 1024), dtype=int32)


#### Building Token embedding and Positional embedding

##### testing the jax embedding

In [124]:
inputs = "Hello how are you"
tokens = tokenizer.encode(inputs).ids
jax_tokens = jnp.asarray([tokens])
jax_pos = jnp.arange(0, len(tokens))


vocab_size = tokenizer.get_vocab_size()
embed_dim = 2
maxlen = len(tokens)
print(maxlen)
rngs = nnx.Rngs(0)

token_embedding = nnx.Embed(num_embeddings = vocab_size, features = embed_dim, rngs = rngs)
positional_embedding = nnx.Embed(num_embeddings = maxlen, features = embed_dim, rngs = rngs)


5


In [115]:
token_embedding, positional_embedding

(Embed( # Param: 20,000 (80.0 KB)
   embedding=Param( # 20,000 (80.0 KB)
     value=Array(shape=(10000, 2), dtype=dtype('float32'))
   ),
   num_embeddings=10000,
   features=2,
   dtype=dtype('float32'),
   param_dtype=float32,
   promote_dtype=<function promote_dtype at 0xff69aa26d6c0>
 ),
 Embed( # Param: 10 (40 B)
   embedding=Param( # 10 (40 B)
     value=Array(shape=(5, 2), dtype=dtype('float32'))
   ),
   num_embeddings=5,
   features=2,
   dtype=dtype('float32'),
   param_dtype=float32,
   promote_dtype=<function promote_dtype at 0xff69aa26d6c0>
 ))

In [116]:
token_embedding(jax_tokens)

Array([[ 0.39644232,  0.5908411 ],
       [ 0.6713258 ,  0.41438612],
       [-0.775721  , -0.47760543],
       [ 0.5454083 ,  0.35636657],
       [ 0.52773136,  0.22185388]], dtype=float32)

In [117]:
positional_embedding(jax_pos)

Array([[-1.727077  , -1.4394435 ],
       [ 0.14534172, -0.24999774],
       [-0.538797  , -0.8333619 ],
       [-0.81191385,  0.21012793],
       [-0.9266888 ,  1.5062807 ]], dtype=float32)

##### Embedding block

In [12]:
class EmbeddingBlock(nnx.Module):
    def __init__(self, vocab_size: int, maxlen: int, embed_dim: int, rngs: nnx.Rngs):
        self.token_embed = nnx.Embed(num_embeddings = vocab_size, features = embed_dim, rngs = rngs)
        self.pos_embed = nnx.Embed(num_embeddings = maxlen, features = embed_dim, rngs = rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        poisitons = jnp.arange(0, x.shape[-1])
        position_embedding = self.pos_embed(poisitons)
        token_embedding = self.token_embed(x)
        return token_embedding + position_embedding

In [13]:
eb = EmbeddingBlock(vocab_size = 10000, maxlen = 1024, embed_dim = 768, rngs = nnx.Rngs(0))

In [14]:
for batch in dataloader:
    out = eb(batch)
    break

In [16]:
out.shape

(32, 1024, 768)

#### Transformer Block

In [65]:
class AttentionBlock(nnx.Module):
    def __init__(self, num_heads: int, embed_dim: int, d_ff: int, rngs: nnx.Rngs, rate: float = 0.1):
        self.mha = nnx.MultiHeadAttention(num_heads = num_heads, in_features = embed_dim, rngs = rngs)
        self.dropout1 = nnx.Dropout(rate = rate, rngs = rngs)
        self.layer_norm1 = nnx.LayerNorm(epsilon = 1e-6, num_features = embed_dim, rngs = rngs)
        self.mlp = nnx.Sequential(
            nnx.Linear(in_features = embed_dim, out_features = d_ff, rngs = rngs),
            nnx.relu,
            nnx.Linear(in_features = d_ff, out_features = embed_dim, rngs = rngs),
            nnx.Dropout(rate = rate, rngs = rngs))
        self.layer_norm2 = nnx.LayerNorm(epsilon = 1e-6, num_features = embed_dim, rngs = rngs)
    
    def __call__(self, inputs, deterministic: bool= True):
        attention_output = self.dropout1(self.mha(inputs_q = inputs, decode = False))
        normalized = self.layer_norm1(inputs + attention_output)
        ffn_output = self.mlp(normalized)
        return self.layer_norm2(normalized + ffn_output)

        

In [61]:
ab = AttentionBlock(num_heads = 8, embed_dim = 768, d_ff = 768 * 4, rngs = nnx.Rngs(1))

In [ ]:
for batch in dataloader:
    eb_out = eb(batch)
    print(ab(eb_out).shape)
    break